In [0]:
%run /Workspace/Users/ashutoshp2@icloud.com/ecommerce_data_pipeline/src/task2_enriched_dims

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

In [0]:
def test_staging_tables_exist():
    tables = [
        "az_ci_de_dbs.ecom_dev.stg_customers",
        "az_ci_de_dbs.ecom_dev.stg_products"
    ]
    results = []
    for table in tables:
        exists = spark.catalog.tableExists(table)
    for res in results:
        print(res)


In [0]:
stg_customers = spark.read.table("az_ci_de_dbs.ecom_dev.stg_customers")
stg_products = spark.read.table("az_ci_de_dbs.ecom_dev.stg_products")

In [0]:
# DQ tests on stg_customers

def test_dq_customers_no_null_ids():
    result = dq_customers(stg_customers)
    assert result.filter(F.col("customer_id").isNull()).count() == 0

def test_dq_customers_id_format():
    result = dq_customers(stg_customers)
    invalid = result.filter(~F.col("customer_id").rlike(r"^[A-Z]{2}-\d{5}$")).count()
    assert invalid == 0, f"Found {invalid} invalid customer_ids after DQ"

def test_dq_customers_name_format():
    result = dq_customers(stg_customers)
    invalid = result.filter(~F.col("customer_name").rlike(r"^[A-Za-z]+( [A-Za-z]+)+$")).count()
    assert invalid == 0, f"Found {invalid} invalid customer_names after DQ"

def test_dq_customers_postal_code_numeric():
    result = dq_customers(stg_customers)
    invalid = result.filter(~F.col("postal_code").rlike(r"^\d+$")).count()
    assert invalid == 0, f"Found {invalid} non-numeric postal_codes after DQ"

def test_dq_customers_country_alphabetic():
    result = dq_customers(stg_customers)
    invalid = result.filter(
        F.col("country").isNotNull() & ~F.col("country").rlike(r"^[A-Za-z\s]+$")
    ).count()
    assert invalid == 0, f"Found {invalid} non-alphabetic countries after DQ"

def test_dq_customers_reduces_or_keeps_count():
    result = dq_customers(stg_customers)
    assert result.count() <= stg_customers.count()
    assert result.count() > 0, "DQ filtered all rows - check rules"

In [0]:
# DQ tests on stg_products

def test_dq_products_no_nulls():
    result = dq_products(stg_products.withColumn("price_per_product", F.col("price_per_product").cast(DoubleType())))
    for col_name in ["product_id", "product_name", "category", "sub_category", "state"]:
        nulls = result.filter(F.col(col_name).isNull()).count()
        assert nulls == 0, f"Found {nulls} nulls in {col_name} after DQ"

def test_dq_products_id_format():
    df = stg_products.withColumn("price_per_product", F.col("price_per_product").cast(DoubleType()))
    result = dq_products(df)
    invalid = result.filter(~F.col("product_id").rlike(r"^[A-Z]{3}-[A-Z]{2}-\d{8}$")).count()
    assert invalid == 0, f"Found {invalid} invalid product_ids after DQ"

def test_dq_products_category_alphabetic():
    df = stg_products.withColumn("price_per_product", F.col("price_per_product").cast(DoubleType()))
    result = dq_products(df)
    invalid = result.filter(~F.col("category").rlike(r"^[A-Za-z\s]+$")).count()
    assert invalid == 0, f"Found {invalid} non-alphabetic categories after DQ"

def test_dq_products_price_non_negative():
    df = stg_products.withColumn("price_per_product", F.col("price_per_product").cast(DoubleType()))
    result = dq_products(df)
    negatives = result.filter(F.col("price_per_product") < 0).count()
    assert negatives == 0, f"Found {negatives} negative prices after DQ"

def test_dq_products_reduces_or_keeps_count():
    df = stg_products.withColumn("price_per_product", F.col("price_per_product").cast(DoubleType()))
    result = dq_products(df)
    assert result.count() <= stg_products.count()
    assert result.count() > 0, "DQ filtered all products - check rules"

In [0]:
# Dedupe tests

def test_dedupe_customers_no_duplicates():
    df = dq_customers(stg_customers)
    result = dedupe(df, ["customer_id"])
    total = result.count()
    distinct_ids = result.select("customer_id").distinct().count()
    assert total == distinct_ids, f"Duplicates remain: {total} rows, {distinct_ids} distinct ids"

def test_dedupe_products_no_duplicates():
    df = stg_products.withColumn("price_per_product", F.col("price_per_product").cast(DoubleType()))
    df = dq_products(df)
    result = dedupe(df)
    total = result.count()
    distinct = result.dropDuplicates().count()
    assert total == distinct, f"Duplicates remain: {total} rows, {distinct} distinct"

def test_dedupe_reduces_or_keeps_count():
    df = dq_customers(stg_customers)
    result = dedupe(df, ["customer_id"])
    assert result.count() <= df.count()

In [0]:
# Apply schema tests

def test_apply_customer_schema_columns():
    df = dq_customers(stg_customers)
    df = dedupe(df, ["customer_id"])
    result = apply_schema(df, customer_schema)
    assert result.columns == [f.name for f in customer_schema.fields]

def test_apply_product_schema_columns():
    df = stg_products.withColumn("price_per_product", F.col("price_per_product").cast(DoubleType()))
    df = dq_products(df)
    df = dedupe(df)
    result = apply_schema(df, product_schema)
    assert result.columns == [f.name for f in product_schema.fields]

def test_apply_schema_no_extra_columns():
    df = dq_customers(stg_customers)
    result = apply_schema(df, customer_schema)
    assert len(result.columns) == len(customer_schema.fields)

In [0]:
# Dim table output tests - built from stg tables using pipeline functions

# Build dim outputs 
def _build_dim_customer_temp():
    df = fill_defaults(stg_customers, configs["Customers"]["defaults"])
    df = dq_customers(df)
    df = dedupe(df, ["customer_id"])
    df = apply_schema(df, customer_schema)
    df = df.withColumn("customer_key", F.monotonically_increasing_id())
    return df

def _build_dim_product_temp():
    df = stg_products.withColumn("price_per_product", F.col("price_per_product").cast(DoubleType()))
    df = fill_defaults(df, configs["Products"]["defaults"])
    df = dq_products(df)
    df = dedupe(df)
    df = apply_schema(df, product_schema)
    df = df.withColumn("product_key", F.monotonically_increasing_id())
    return df

dim_customer_df = _build_dim_customer_temp()
dim_product_df = _build_dim_product_temp()

def test_dim_customer_count():
    assert dim_customer_df.count() > 0, "dim_customer is empty"
    assert dim_customer_df.count() <= stg_customers.count(), "dim_customer has more rows than stg"

def test_dim_product_count():
    assert dim_product_df.count() > 0, "dim_product is empty"
    assert dim_product_df.count() <= stg_products.count(), "dim_product has more rows than stg"

def test_dim_customer_has_surrogate_key():
    assert "customer_key" in dim_customer_df.columns
    assert dim_customer_df.filter(F.col("customer_key").isNull()).count() == 0

def test_dim_product_has_surrogate_key():
    assert "product_key" in dim_product_df.columns
    assert dim_product_df.filter(F.col("product_key").isNull()).count() == 0

In [0]:
# Run all tests
def main_dims_test():
    test_functions = [
        # stg table exists
        test_staging_tables_exist,
        # DQ customers
        test_dq_customers_no_null_ids,
        test_dq_customers_id_format,
        test_dq_customers_name_format,
        test_dq_customers_postal_code_numeric,
        test_dq_customers_country_alphabetic,
        test_dq_customers_reduces_or_keeps_count,
        # DQ products
        test_dq_products_no_nulls,
        test_dq_products_id_format,
        test_dq_products_category_alphabetic,
        test_dq_products_price_non_negative,
        test_dq_products_reduces_or_keeps_count,
        # Dedupe
        test_dedupe_customers_no_duplicates,
        test_dedupe_products_no_duplicates,
        test_dedupe_reduces_or_keeps_count,
        # Apply schema
        test_apply_customer_schema_columns,
        test_apply_product_schema_columns,
        test_apply_schema_no_extra_columns,
        # Dim table counts
        test_dim_customer_count,
        test_dim_product_count,
        test_dim_customer_has_surrogate_key,
        test_dim_product_has_surrogate_key,
    ]

    passed, failed = 0, 0
    for fn in test_functions:
        try:
            fn()
            passed += 1
            print(f"  [PASSED] {fn.__name__}")
        except Exception as e:
            failed += 1
            print(f"  [FAILED] {fn.__name__}: {e}")

    print(f"\nResults: {passed} passed, {failed} failed, {passed + failed} total")
